# RESIBO research restoration handoff — Colab
This notebook processes an exported production face job and returns enhanced images, AdaFace embeddings, and ranked cosine candidates. It does not make identity decisions.

**OSDFace order:** upload handoff ZIP → model setup → restart session → upload request ZIP → process → download result.
Use a GPU for OSDFace. Source images are preserved; 512x512 model inputs are returned as separate artifacts.


In [ ]:
# Ito yung cell na naglo-load ng handoff Python/SQL files into Colab.
# Upload RESIBO_Research_Handoff_Annotated.zip, not the dataset or request ZIP yet.
%pip -q install "Pillow>=11,<13"
from google.colab import files
from pathlib import Path, PurePosixPath
import zipfile, os
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith('.zip')]
assert len(archives) == 1, 'Upload exactly one handoff project ZIP'
with zipfile.ZipFile(archives[0]) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)), 'Duplicate ZIP paths'
    for name in names:
        path = PurePosixPath(name)
        assert not path.is_absolute() and '..' not in path.parts and '\\' not in name, 'Unsafe ZIP path'
    archive.extractall('/content')
PROJECT = Path('/content/RESIBO_Research_Handoff_Annotated')
if not (PROJECT/'colab_setup_osdface.py').is_file():
    PROJECT = Path('/content')
assert (PROJECT/'colab_setup_osdface.py').is_file(), 'Wrong project ZIP'
os.chdir(PROJECT)


## Model setup
The setup preserves Colab's Torch/CUDA pair, installs pinned dependencies, downloads OSDFace and AdaFace weights, verifies the strict LoRA assignment, and performs GPU preflight checks.

Select `osdface` for model restoration. Select `resize512_smoke` to test only local-to-Colab transfer without loading a model.


In [ ]:
# Ito yung model setup cell. It does not create or edit the laptop SQLite file.
import json, subprocess, sys
MODE = 'osdface'
(PROJECT/'worker_mode.json').write_text(json.dumps({'mode': MODE}))
subprocess.run([sys.executable, '-u', str(PROJECT/'colab_setup_osdface.py')], check=True)
print('PREFLIGHT PASSED: restart the runtime, then run the production job cell.')


## After restart: process one exported job
Colab files persist through **Restart session**, but Python variables do not. This cell rebuilds its paths and reads the saved mode.
Upload a request ZIP created by `python app.py export ...` on the laptop.


In [ ]:
# Ito yung job cell: request images -> OSDFace/resize worker -> results ZIP.
from google.colab import files
from pathlib import Path
import os, json, sys, subprocess, uuid, hashlib
PROJECT = Path('/content/RESIBO_Research_Handoff_Annotated')
if not (PROJECT/'colab_worker.py').is_file():
    PROJECT = Path('/content')
os.chdir(PROJECT)
MODE = json.loads((PROJECT/'worker_mode.json').read_text())['mode']
print('Upload ONE request ZIP exported by the local application.')
uploaded = files.upload()
requests = [name for name in uploaded if name.endswith('.zip')]
assert len(requests) == 1, 'Upload exactly one request ZIP'
request = PROJECT/requests[0]
result = PROJECT/('results_' + uuid.uuid4().hex[:8] + '.zip')
log_file = result.with_suffix('.log')
command = [sys.executable, '-u', str(PROJECT/'colab_worker.py'), '--request', str(request), '--out', str(result)]
sys.path.insert(0, str(PROJECT))
from osdface_adapter import sha_file
setup_file = Path('/content/resibo_setup/ready.json')
assert setup_file.is_file(), 'Model preflight is missing; rerun setup.'
setup = json.loads(setup_file.read_text())
checkpoint = Path(setup['repo'])/'pretrained'/'pytorch_lora_weights.safetensors'
command += ['--method', 'external_restoration', '--adapter', 'osdface_adapter:restore',
            '--checkpoint-sha256', sha_file(checkpoint),
            '--embedding-adapter', 'adaface_adapter:embed']
with log_file.open('w') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end=''); log.write(line); log.flush()
    exit_code = process.wait()
assert exit_code == 0 and result.is_file(), f'Worker failed. Inspect {log_file}; no successful ZIP was returned.'
print('Complete result ZIP:', result.name)
files.download(str(result))
files.download(str(log_file))


## Return to VSCode
Move the result ZIP and log under `exchange/`. Import with `python app.py import --zip exchange/ACTUAL_RESULTS_NAME.zip`, then run `python app.py verify` and `python app.py status`.

The local database stores source and restored image BLOBs; OSDFace mode additionally stores each resized model input. The result manifest records model/checkpoint hashes, source revision, versions, preparation, and seed. The master SQLite file remains local.

The new bridge requires an actual GPU acceptance run before it can be called GPU-validated. The package's automated tests use synthetic adapters for transfer validation.
